# Project 1 : Eliminating Child Care Deserts in New York City


In [1]:
from gurobipy import Model, GRB, quicksum, Env
import gurobipy as gp
import pandas as pd
import numpy as np
import math
from scipy.spatial import cKDTree

## Uploading & cleaning data 

In [2]:
child_care_regulated  = pd.read_csv('child_care_regulated_nyc.csv')
population            = pd.read_csv('population_nyc.csv')
avg_individual_income = pd.read_csv('avg_individual_income_nyc.csv')
employment_rate       = pd.read_csv('employment_rate_nyc.csv')
potential_locations   = pd.read_csv('potential_locations_nyc.csv')

In [3]:
# Data cleaning : child_care_regulated
child_care_regulated.info()
# Missing values for latitude / longitude
child_care_regulated[child_care_regulated['latitude'].isna()]
child_care_regulated = child_care_regulated.drop(columns='Unnamed: 0', errors='ignore')

# Keep only active facilities (remove suspended / pending revocation)
ACTIVE_STATUSES = ['License', 'Registration']
existing_facilities = child_care_regulated[
    child_care_regulated['facility_status'].isin(ACTIVE_STATUSES)
].copy().reset_index(drop=True)

# Compute 0-5 years capacity correctly.
# infant_capacity and toddler_capacity are all-zero in this dataset.
# children_capacity is a mixed 0-12 bucket used by FDC and GFDC facilities;
# we prorate it by the citywide share of 0-5 children among 0-12 children.
# SACC facilities are school-age only, so no proration is applied.
_pop_tmp = pd.read_csv('population_nyc.csv')[['zipcode', '-5', '6-12']]
_pop_tmp.columns = ['zipcode', 'pop_0_5', 'pop_6_12']
_pop_tmp = _pop_tmp[(_pop_tmp['pop_0_5'] + _pop_tmp['pop_6_12']) > 0]
CITY_RATIO_0_5 = (_pop_tmp['pop_0_5'].sum() /
                  (_pop_tmp['pop_0_5'] + _pop_tmp['pop_6_12']).sum())

existing_facilities['0-5 years capacity'] = existing_facilities.apply(
    lambda r: (
        r['infant_capacity'] + r['toddler_capacity'] + r['preschool_capacity']
        if r['program_type'] == 'SACC'
        else r['infant_capacity'] + r['toddler_capacity'] + r['preschool_capacity']
             + CITY_RATIO_0_5 * r['children_capacity']
    ), axis=1
)
existing_facilities = existing_facilities.rename(
    columns={'school_age_capacity': '5-12 years capacity'}
)

# Impute missing coordinates using the zip-code centroid from potential_locations
_zip_coords = pd.read_csv('potential_locations_nyc.csv').drop(columns='Unnamed: 0', errors='ignore')
_zip_centroid = _zip_coords.groupby('zipcode')[['latitude', 'longitude']].mean()

def _fill_coords(row):
    if pd.isna(row['latitude']) or pd.isna(row['longitude']):
        if row['zipcode'] in _zip_centroid.index:
            return _zip_centroid.loc[row['zipcode'], 'latitude'], _zip_centroid.loc[row['zipcode'], 'longitude']
    return row['latitude'], row['longitude']

_filled = existing_facilities.apply(_fill_coords, axis=1, result_type='expand')
existing_facilities['latitude']  = _filled[0]
existing_facilities['longitude'] = _filled[1]

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7740 entries, 0 to 7739
Data columns (total 16 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Unnamed: 0            7740 non-null   int64  
 1   facility_id           7740 non-null   int64  
 2   program_type          7740 non-null   object 
 3   facility_status       7740 non-null   object 
 4   facility_name         7740 non-null   object 
 5   city                  7740 non-null   object 
 6   zipcode               7740 non-null   int64  
 7   school_district_name  7725 non-null   object 
 8   infant_capacity       7740 non-null   int64  
 9   toddler_capacity      7740 non-null   int64  
 10  preschool_capacity    7740 non-null   int64  
 11  school_age_capacity   7740 non-null   int64  
 12  children_capacity     7740 non-null   int64  
 13  total_capacity        7740 non-null   int64  
 14  latitude              7571 non-null   float64
 15  longitude            

In [4]:
# Data cleaning : population
population.info()
population.head()
# We only care about the age range 0-12
population = population[['zipcode', '-5', '6-12']]

# Drop zip codes with zero children (commercial-only / PO-box zip codes;
# these cannot be child care deserts)
population = population[(population['-5'] + population['6-12']) > 0].copy()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 211 entries, 0 to 210
Data columns (total 21 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   Unnamed: 0  211 non-null    int64
 1   zipcode     211 non-null    int64
 2   Total       211 non-null    int64
 3   -5          211 non-null    int64
 4   6-12        211 non-null    int64
 5   13-14       211 non-null    int64
 6   15-19       211 non-null    int64
 7   20-24       211 non-null    int64
 8   25-29       211 non-null    int64
 9   30-34       211 non-null    int64
 10  35-39       211 non-null    int64
 11  40-44       211 non-null    int64
 12  45-49       211 non-null    int64
 13  50-54       211 non-null    int64
 14  55-59       211 non-null    int64
 15  60-64       211 non-null    int64
 16  65-69       211 non-null    int64
 17  70-74       211 non-null    int64
 18  75-79       211 non-null    int64
 19  80-84       211 non-null    int64
 20  85+         211 non-null    int6

In [5]:
# Verification : avg_individual_income
avg_individual_income.info()
# We check that we don't have any duplicate in the zipcode
avg_individual_income['zipcode'].duplicated().any()
avg_individual_income = avg_individual_income.drop(columns='Unnamed: 0')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 179 entries, 0 to 178
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Unnamed: 0      179 non-null    int64  
 1   zipcode         179 non-null    int64  
 2   average income  179 non-null    float64
dtypes: float64(1), int64(2)
memory usage: 4.3 KB


In [6]:
employment_rate = employment_rate.drop(columns='Unnamed: 0')
employment_rate.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 179 entries, 0 to 178
Data columns (total 2 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   zipcode          179 non-null    int64  
 1   employment rate  179 non-null    float64
dtypes: float64(1), int64(1)
memory usage: 2.9 KB


In [7]:
# Verification : potential_locations
potential_locations = potential_locations.drop(columns='Unnamed: 0')
potential_locations.info()
potential_locations.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31100 entries, 0 to 31099
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   zipcode    31100 non-null  int64  
 1   latitude   31100 non-null  float64
 2   longitude  31100 non-null  float64
dtypes: float64(2), int64(1)
memory usage: 729.0 KB


,zipcode,latitude,longitude
0,10001,40.741893,-74.000140
1,10001,40.752007,-74.005436
2,10001,40.750545,-73.997147
3,10001,40.744080,-74.001932
4,10001,40.748690,-73.999341


### Imputation of missing zip codes


In [8]:
# Some zip codes appear in population but not in income / employment.
# These include zip 11249 (9 610 children) and 11430 (37 children), which the
# original inner merge silently drops. We impute using inverse-distance-weighted
# K-Nearest-Neighbour (k=3) based on geographic centroids from potential_locations.

def _knn_impute(lat, lon, df, col, k=3):
    tree = cKDTree(df[['latitude', 'longitude']].values)
    dists, idxs = tree.query([[lat, lon]], k=k)
    return float(np.average(df[col].values[idxs[0]], weights=1.0 / (dists[0] + 1e-9)))

_zip_centroids = potential_locations.groupby('zipcode')[['latitude', 'longitude']].mean().reset_index()
_known_inc = _zip_centroids.merge(avg_individual_income, on='zipcode')
_known_emp = _zip_centroids.merge(employment_rate,       on='zipcode')

_missing = sorted(set(population['zipcode']) - set(avg_individual_income['zipcode']))
print(f'Zip codes needing imputation: {_missing}')

_extra_inc, _extra_emp = [], []
for _z in _missing:
    _row = _zip_centroids[_zip_centroids['zipcode'] == _z]
    if _row.empty:
        _lat = _known_inc['latitude'].mean()
        _lon = _known_inc['longitude'].mean()
    else:
        _lat, _lon = float(_row.iloc[0]['latitude']), float(_row.iloc[0]['longitude'])
    _imp_inc = _knn_impute(_lat, _lon, _known_inc, 'average income')
    _imp_emp = _knn_impute(_lat, _lon, _known_emp, 'employment rate')
    _extra_inc.append({'zipcode': _z, 'average income': _imp_inc})
    _extra_emp.append({'zipcode': _z, 'employment rate': _imp_emp})
    print(f'  Zip {_z}: average income={_imp_inc:,.0f}, employment rate={_imp_emp:.4f}')

avg_individual_income = pd.concat(
    [avg_individual_income, pd.DataFrame(_extra_inc)], ignore_index=True)
employment_rate = pd.concat(
    [employment_rate, pd.DataFrame(_extra_emp)], ignore_index=True)

Zip codes needing imputation: [11249, 11430]
  Zip 11249: average income=53,567, employment rate=0.4736
  Zip 11430: average income=53,542, employment rate=0.4778


### Creation of the childcare desert area subset


In [9]:
# Creation of the zip code dataset with the employment rate, the number of children
# (age range 0-5 and 6-12)
# FIX 3 cont.: use outer-safe merge so imputed zips are not lost
areas = pd.merge(avg_individual_income, employment_rate, on='zipcode', how='inner')
areas = pd.merge(areas, population,            on='zipcode', how='inner')
areas.info()
areas.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 180 entries, 0 to 179
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   zipcode          180 non-null    int64  
 1   average income   180 non-null    float64
 2   employment rate  180 non-null    float64
 3   -5               180 non-null    int64  
 4   6-12             180 non-null    int64  
dtypes: float64(2), int64(3)
memory usage: 7.2 KB


,zipcode,average income,employment rate,-5,6-12
0,10001,102878.033603,0.595097,744,1255
1,10002,59604.041165,0.520662,2142,4645
2,10003,114273.049645,0.497244,1440,1510
3,10004,132004.310345,0.506661,433,262
4,10005,121437.713311,0.665833,484,318


In [10]:
# Calculating the number of slots for each zip code.
# Aggregate from existing_facilities (active-only, correct 0-5 estimate)
# and use total_capacity as the definitive slot count rather than summing
# sub-columns that miss children_capacity.
child_care_regulated_aggregate = existing_facilities.groupby('zipcode').agg(
    total_capacity       =('total_capacity',       'sum'),
    **{'0-5 years capacity': pd.NamedAgg('0-5 years capacity', 'sum')},
    **{'5-12 years capacity': pd.NamedAgg('5-12 years capacity', 'sum')},
).reset_index()

child_care_regulated_aggregate.info()
child_care_regulated_aggregate.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 179 entries, 0 to 178
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   zipcode              179 non-null    int64  
 1   total_capacity       179 non-null    int64  
 2   0-5 years capacity   179 non-null    float64
 3   5-12 years capacity  179 non-null    int64  
dtypes: float64(1), int64(3)
memory usage: 5.7 KB


,zipcode,total_capacity,0-5 years capacity,5-12 years capacity
count,179.000000,179.000000,179.000000,179.000000
mean,10801.497207,1773.748603,165.111555,1384.418994
std,575.177846,1511.200330,230.897252,1078.495294
min,10001.000000,16.000000,0.000000,4.000000
25%,10291.500000,600.000000,34.828444,510.500000
50%,11106.000000,1444.000000,80.567002,1157.000000
75%,11359.000000,2553.500000,190.865640,1976.000000
max,11694.000000,6984.000000,1214.799331,4838.000000


In [11]:
# Adding the 0-5 years, 5-12 years, and total capacities to the areas dataset
areas = areas.merge(
    child_care_regulated_aggregate[['zipcode', 'total_capacity',
                                    '5-12 years capacity', '0-5 years capacity']],
    on='zipcode', how='left'
)
areas[['total_capacity', '5-12 years capacity', '0-5 years capacity']] = (
    areas[['total_capacity', '5-12 years capacity', '0-5 years capacity']].fillna(0)
)

In [12]:
# Filtering on child care desert for high demand areas
# Use total_capacity (all existing slots) for the desert threshold,
# not just '5-12 years capacity' + '0-5 years capacity' which misses
# children_capacity.
child_care_desert_high_demand = areas[
    ((areas['employment rate'] >= 0.6) | (areas['average income'] <= 60000)) &
    (areas['total_capacity'] <= 0.5 * (areas['-5'] + areas['6-12']))
].copy()
child_care_desert_high_demand['demand_type'] = 'high_demand'
child_care_desert_high_demand.info()
child_care_desert_high_demand.head()

<class 'pandas.core.frame.DataFrame'>
Index: 91 entries, 4 to 179
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   zipcode              91 non-null     int64  
 1   average income       91 non-null     float64
 2   employment rate      91 non-null     float64
 3   -5                   91 non-null     int64  
 4   6-12                 91 non-null     int64  
 5   total_capacity       91 non-null     float64
 6   5-12 years capacity  91 non-null     float64
 7   0-5 years capacity   91 non-null     float64
 8   demand_type          91 non-null     object 
dtypes: float64(5), int64(3), object(1)
memory usage: 7.1+ KB


,zipcode,average income,employment rate,-5,6-12,total_capacity,5-12 years capacity,0-5 years capacity,demand_type
4,10005,121437.713311,0.665833,484,318,39.0,39.0,0.000000,high_demand
14,10017,118061.722913,0.777302,193,293,107.0,107.0,0.000000,high_demand
15,10018,108746.189024,0.757593,242,280,0.0,0.0,0.000000,high_demand
16,10019,107014.967791,0.617079,1099,949,591.0,543.0,20.141751,high_demand
25,10029,48339.672323,0.452033,3965,5301,3358.0,3000.0,150.223890,high_demand


In [13]:
# Filtering on child care desert for low demand areas
# Use total_capacity for the same reason as above.
child_care_desert_low_demand = areas[
    ((areas['employment rate'] < 0.6) & (areas['average income'] > 60000)) &
    (areas['total_capacity'] <= (areas['-5'] + areas['6-12']) / 3)
].copy()
child_care_desert_low_demand['demand_type'] = 'low_demand'

# Concatenating all the desert areas
ccdesert_areas = pd.concat(
    [child_care_desert_low_demand, child_care_desert_high_demand]
).reset_index(drop=True)
ccdesert_areas.info()
ccdesert_areas.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 162 entries, 0 to 161
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   zipcode              162 non-null    int64  
 1   average income       162 non-null    float64
 2   employment rate      162 non-null    float64
 3   -5                   162 non-null    int64  
 4   6-12                 162 non-null    int64  
 5   total_capacity       162 non-null    float64
 6   5-12 years capacity  162 non-null    float64
 7   0-5 years capacity   162 non-null    float64
 8   demand_type          162 non-null    object 
dtypes: float64(5), int64(3), object(1)
memory usage: 11.5+ KB


,zipcode,average income,employment rate,-5,6-12,total_capacity,5-12 years capacity,0-5 years capacity,demand_type
0,10001,102878.033603,0.595097,744,1255,609.0,585.0,10.070875,low_demand
1,10007,138853.904282,0.528910,605,596,284.0,284.0,0.000000,low_demand
2,10010,116272.698810,0.492749,1422,2066,234.0,234.0,0.000000,low_demand
3,10012,111131.786340,0.538273,613,364,24.0,6.0,7.553156,low_demand
4,10013,106023.044693,0.486198,1249,1796,435.0,423.0,5.035438,low_demand


### NYC under-5 policy parameters


In [14]:
# The project requires that, for every zip code, the number of slots
# available to children aged 0-5 must be at least 2/3 of the population aged 0-5.
# Precompute this requirement across all zip codes (not just deserts) so it can
# be enforced as a constraint in both models.
areas['under5_min_slots'] = (2 / 3) * areas['-5']
areas['under5_deficit']   = np.maximum(
    0, areas['under5_min_slots'] - areas['0-5 years capacity']
)
print(f"Zip codes with under-5 slot shortfall: "
      f"{(areas['under5_deficit'] > 0).sum()} / {len(areas)}")
areas[['zipcode', '-5', '0-5 years capacity', 'under5_min_slots', 'under5_deficit']].head(10)

Zip codes with under-5 slot shortfall: 179 / 180


,zipcode,-5,0-5 years capacity,under5_min_slots,under5_deficit
0,10001,744,10.070875,496.000000,485.929125
1,10002,2142,101.084721,1428.000000,1326.915279
2,10003,1440,0.000000,960.000000,960.000000
3,10004,433,0.000000,288.666667,288.666667
4,10005,484,0.000000,322.666667,322.666667
5,10006,128,14.000000,85.333333,71.333333
6,10007,605,0.000000,403.333333,403.333333
7,10009,1896,62.479699,1264.000000,1201.520301
8,10010,1422,0.000000,948.000000,948.000000
9,10011,1209,44.517719,806.000000,761.482281


###  Creation of the potential locations for construction of facilities 

In [15]:
print(potential_locations['zipcode'].unique().size)
# We have 311 zip codes with potential locations across NYC.

# The problem states new facilities can
# be built ANYWHERE in NYC, so we use all potential locations, not only those
# inside desert zip codes.
locations = potential_locations.reset_index().rename(columns={'index': 'location_id'})
print(f"Total potential locations: {len(locations)}")
locations.head()

311
Total potential locations: 31100


,location_id,zipcode,latitude,longitude
0,0,10001,40.741893,-74.000140
1,1,10001,40.752007,-74.005436
2,2,10001,40.750545,-73.997147
3,3,10001,40.744080,-74.001932
4,4,10001,40.748690,-73.999341


### Creation of the function which calculates the distance between 2 locations 

In [16]:
def distance(lat1, lon1, lat2, lon2):
    # Return the distance in miles between 2 points defined with their
    # latitude and longitude using the Haversine formula
    R = 3958.8  # radius of the Earth in miles
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi    = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    # Haversine formula
    a = math.sin(dphi / 2)**2 + \
        math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    return R * c

In [17]:
# TEST
point1 = potential_locations.iloc[1]
point2 = child_care_regulated.iloc[1]
lat1, long1 = point1.latitude, point1.longitude
lat2, long2 = point2.latitude, point2.longitude
distance(lat1, long1, lat2, long2)

1.8151195447747606

### Creation of the dataset containing the information about the different type of new facilities we can build.

In [18]:
# New slots for children under 5 require $100 per slot for specialised
# equipment. The base costs from Table 1 must be increased accordingly.
EQUIP_COST_PER_0_5_SLOT = 100  # $/slot for 0-5 equipment

data = {
    'facility_type':   [1,   2,   3],
    'slots total':     [100, 200, 400],
    'slots 0-5 years': [50,  100, 200],
    'base_cost':       [65000, 95000, 115000],
}
facilities_type = pd.DataFrame(data)
facilities_type['equip_cost'] = facilities_type['slots 0-5 years'] * EQUIP_COST_PER_0_5_SLOT
facilities_type['cost']       = facilities_type['base_cost'] + facilities_type['equip_cost']
print(facilities_type)

   facility_type  slots total  slots 0-5 years  base_cost  equip_cost    cost
0              1          100               50      65000        5000   70000
1              2          200              100      95000       10000  105000
2              3          400              200     115000       20000  135000


## Model 1

Objective: Minimize total funding (expansion + new facilities) to eliminate child care deserts in every zip code.

- **Expansion**: Each existing facility can add up to min(20% of current capacity, 500 - current) slots. Facilities with ≥500 slots excluded.
- **Expansion cost**: (20,000 + 200 × n_f) × (x_f / n_f) + $100 per 0-5 slot.
  
- **New facilities**: Base cost from Table 1; $100/slot only for ACTUAL 0-5 slots configured (50/100/200 is max per facility type).
- **Note**: Expansion is aggregated by zip (sum of facility capacities) to fit Gurobi license limits; per-slot cost uses zip-average.


In [19]:
# ========== Model 1: Prepare expansion capacity by zip ==========
# Per PDF: expansion limit min(20% of current, 500-current); exclude n>=500
facilities_df = existing_facilities[['facility_id', 'zipcode', 'total_capacity', '0-5 years capacity', '5-12 years capacity']].copy()
facilities_df = facilities_df.rename(columns={
    'total_capacity': 'n_total',
    '0-5 years capacity': 'n_05',
    '5-12 years capacity': 'n_512'
})
facilities_df['max_expand'] = facilities_df.apply(
    lambda r: min(0.2 * r['n_total'], max(0, 500 - r['n_total'])) if r['n_total'] < 500 else 0,
    axis=1
)

# Aggregate by zip to keep model size manageable (Gurobi license limits)
_exp_by_zip = facilities_df[facilities_df['max_expand'] > 0].groupby('zipcode').agg(
    max_expand_total=('max_expand', 'sum'),
    total_n=('n_total', 'sum'),
    count=('facility_id', 'count')
).reset_index()
# Per-slot expansion cost: (20000+200*n)/n = 20000/n + 200. Use zip avg n.
_exp_by_zip['n_avg'] = _exp_by_zip['total_n'] / _exp_by_zip['count']
_exp_by_zip['cost_per_slot'] = 20000 / _exp_by_zip['n_avg'] + 200
expand_cap = _exp_by_zip.set_index('zipcode').to_dict('index')
print(f"Zips with expansion capacity: {len(expand_cap)}")
print(_exp_by_zip.head())

Zips with expansion capacity: 178
   zipcode  max_expand_total  total_n  count       n_avg  cost_per_slot
0    10001             121.8      609      9   67.666667     495.566502
1    10002             944.8     4724     56   84.357143     437.087214
2    10003             196.2     1995      7  285.000000     270.175439
3    10004              52.6      263      2  131.500000     352.091255
4    10005               7.8       39      1   39.000000     712.820513


In [20]:
# ========== Model 1: Required slots per zip ==========
# Desert: slots <= threshold. To eliminate, need slots > threshold, i.e. slots >= floor(threshold) + 1
areas['pop_0_12'] = areas['-5'] + areas['6-12']
areas['high_demand'] = (areas['employment rate'] >= 0.6) | (areas['average income'] <= 60000)
areas['theta_total'] = np.where(areas['high_demand'], 0.5, 1/3)
areas['min_total_slots'] = np.floor(areas['theta_total'] * areas['pop_0_12']).astype(int) + 1
# For strict "not desert": slots must be > threshold, so min = floor(threshold) + 1 when fractional
# Using ceil ensures we exceed the desert boundary
areas['min_05_slots'] = np.ceil(2/3 * areas['-5']).astype(int)

# Index sets for Gurobi
zips = areas['zipcode'].tolist()

# Facility types for new construction
ft_list = facilities_type['facility_type'].tolist()

In [21]:
# ========== Model 1: Gurobi optimization (zip-aggregated expansion for license limits) ==========
m1 = Model('ChildCare_Model1')
m1.setParam('OutputFlag', 1)

# Decision variables: zip-level expansion (aggregated)
x_expand = {}
x_expand_05 = {}
for z in zips:
    cap = expand_cap.get(z, {'max_expand_total': 0, 'cost_per_slot': 400})
    max_x = cap.get('max_expand_total', 0)
    x_expand[z] = m1.addVar(lb=0, ub=max_x, name=f'x_{z}')
    x_expand_05[z] = m1.addVar(lb=0, ub=max_x, name=f'x05_{z}')

# New facilities: y[z, s] = number of type-s facilities built in zip z
y_new = {(z, s): m1.addVar(lb=0, ub=GRB.INFINITY, vtype=GRB.INTEGER, name=f'y_{z}_{s}')
         for z in zips for s in ft_list}

# u_05_new[z, s] = actual 0-5 slots configured from new type-s facilities in zip z
# (50/100/200 is MAX; we only pay $100 for slots we actually configure as 0-5)
u_05_new = {(z, s): m1.addVar(lb=0, ub=GRB.INFINITY, name=f'u05_{z}_{s}')
            for z in zips for s in ft_list}

m1.update()

Restricted license - for non-production use only - expires 2027-11-29
Set parameter OutputFlag to value 1


In [22]:
# Constraints: x_expand_05 <= x_expand (0-5 portion cannot exceed total expansion)
for z in zips:
    m1.addConstr(x_expand_05[z] <= x_expand[z], name=f'split_{z}')

# u_05_new[z,s] <= y_new[z,s] * max_0_5[s] (actual 0-5 configured cannot exceed max per facility)
ft_dict = facilities_type.set_index('facility_type').to_dict('index')
for z in zips:
    for s in ft_list:
        max_05 = ft_dict[s]['slots 0-5 years']
        m1.addConstr(u_05_new[(z, s)] <= y_new[(z, s)] * max_05, name=f'u05_cap_{z}_{s}')

# Constraint: Total slots per zip >= min_total_slots (eliminate desert)
# 0-5 slots: use u_05_new (actual configured), not full max
areas_dict = areas.set_index('zipcode').to_dict('index')

for z in zips:
    curr_total = areas_dict[z]['total_capacity']
    curr_05 = areas_dict[z]['0-5 years capacity']
    min_total = areas_dict[z]['min_total_slots']
    min_05 = areas_dict[z]['min_05_slots']

    exp_total = x_expand[z]
    exp_05 = x_expand_05[z]
    new_total = quicksum(y_new[(z, s)] * ft_dict[s]['slots total'] for s in ft_list)
    new_05 = quicksum(u_05_new[(z, s)] for s in ft_list)  # actual 0-5 configured

    m1.addConstr(curr_total + exp_total + new_total >= min_total, name=f'total_{z}')
    m1.addConstr(curr_05 + exp_05 + new_05 >= min_05, name=f'under5_{z}')

m1.update()

In [23]:
# Objective: minimize total cost
# Expansion cost (zip-aggregated): cost_per_slot * x + $100 per actual 0-5 slot
cost_expansion = quicksum(
    (expand_cap.get(z, {}).get('cost_per_slot', 400) * x_expand[z] + 100 * x_expand_05[z])
    for z in zips
)

# New facility: base cost + $100 per ACTUAL 0-5 slots configured (not max)
cost_new_base = quicksum(y_new[(z, s)] * ft_dict[s]['base_cost'] for z in zips for s in ft_list)
cost_new_equip = 100 * quicksum(u_05_new[(z, s)] for z in zips for s in ft_list)

m1.setObjective(cost_expansion + cost_new_base + cost_new_equip, GRB.MINIMIZE)
m1.optimize()

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[arm] - Darwin 24.6.0 24G517)

CPU model: Apple M4 Pro
Thread count: 14 physical cores, 14 logical processors, using up to 14 threads

Optimize a model with 1080 rows, 1440 columns and 2880 nonzeros (Min)
Model fingerprint: 0x2dc949d5
Model has 1440 linear objective coefficients
Variable types: 900 continuous, 540 integer (0 binary)
Coefficient statistics:
  Matrix range     [1e+00, 4e+02]
  Objective range  [1e+02, 1e+05]
  Bounds range     [3e+00, 1e+03]
  RHS range        [2e+01, 1e+04]

Found heuristic solution: objective 3.306685e+08
Presolve removed 1074 rows and 1432 columns
Presolve time: 0.06s
Presolved: 6 rows, 8 columns, 16 nonzeros
Found heuristic solution: objective 2.157469e+08
Variable types: 5 continuous, 3 integer (0 binary)

Root relaxation: objective 2.150770e+08, 6 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntIn

In [24]:
# ========== Model 1: Results ==========
if m1.Status == GRB.OPTIMAL:
    print(f"Optimal total cost: ${m1.ObjVal:,.0f}")
    total_expand = sum(x_expand[z].X for z in zips)
    total_new_slots = sum(y_new[(z,s)].X * ft_dict[s]['slots total'] for z in zips for s in ft_list)
    total_new_05 = sum(u_05_new[(z,s)].X for z in zips for s in ft_list)
    n_new_fac = sum(int(y_new[(z,s)].X) for z in zips for s in ft_list)
    print(f"Total expansion slots: {total_expand:,.0f}")
    print(f"Total new facility slots: {total_new_slots:,.0f} ({n_new_fac} new facilities)")
    print(f"Actual 0-5 slots configured (new): {total_new_05:,.0f} (equip cost: ${100*total_new_05:,.0f})")

    # Summary: zips with expansion
    exp_by_zip = [(z, x_expand[z].X) for z in zips if x_expand[z].X > 0]
    if exp_by_zip:
        print(f"\nZips with expansion: {len(exp_by_zip)} (sample): {exp_by_zip[:5]}")
    # New facilities by zip
    new_by_zip = [(z, s, int(y_new[(z,s)].X), u_05_new[(z,s)].X) for z in zips for s in ft_list if y_new[(z,s)].X > 0]
    if new_by_zip:
        print(f"\nNew facilities (sample):")
        for (z, s, cnt, u05) in new_by_zip[:10]:
            slots = ft_dict[s]['slots total'] * cnt
            print(f"  Zip {z}: {cnt} x type-{s} ({slots} total slots, {u05:.0f} configured 0-5)")
else:
    print(f"Model status: {m1.Status}")

Optimal total cost: $215,083,191
Total expansion slots: 17,400
Total new facility slots: 602,600 (1518 new facilities)
Actual 0-5 slots configured (new): 299,919 (equip cost: $29,991,912)

Zips with expansion: 136 (sample): [(10001, 85.92912471858722), (10002, 926.9152789283446), (10003, 160.0), (10004, 39.0), (10006, 22.0)]

New facilities (sample):
  Zip 10001: 2 x type-3 (800 total slots, 400 configured 0-5)
  Zip 10002: 2 x type-3 (800 total slots, 400 configured 0-5)
  Zip 10003: 4 x type-3 (1600 total slots, 800 configured 0-5)
  Zip 10004: 1 x type-1 (100 total slots, 50 configured 0-5)
  Zip 10004: 1 x type-3 (400 total slots, 200 configured 0-5)
  Zip 10005: 2 x type-3 (800 total slots, 323 configured 0-5)
  Zip 10006: 1 x type-1 (100 total slots, 50 configured 0-5)
  Zip 10007: 2 x type-3 (800 total slots, 400 configured 0-5)
  Zip 10009: 5 x type-3 (2000 total slots, 1000 configured 0-5)
  Zip 10010: 5 x type-3 (2000 total slots, 948 configured 0-5)


## Model 2

OVERVIEW: Realistic Capacity Expansion and Location

Objective: Minimize total funding (expansion + new facilities) to eliminate child care deserts across all NYC zip codes.

Expansion Constraints: Each existing facility can add up to a strict maximum of 20% of its current capacity.

Expansion Cost: Piecewise continuous cost function based on the expansion tier, plus an additional $100 for every specific 0-5 years slot added:

Tier 1 (0-10%): (20,000 + 200 × n_f) × (x_f / n_f) + $100 × (0-5 slots)

Tier 2 (10-15%): (20,000 + 400 × n_f) × (x_f / n_f) + $100 × (0-5 slots)

Tier 3 (15-20%): (20,000 + 1000 × n_f) × (x_f / n_f) + $100 × (0-5 slots)

New Facilities: Base cost from Table 1 (Small/Medium/Large) + $100 per actual 0-5 slot configured.

Distance Limit: NO two facilities (new vs. new, or new vs. existing) can be located within 0.06 miles of each other STRICTLY WITHIN each zip code area
.

In [25]:
# Cell 1: Imports & License Setup
import os
import math
from dataclasses import dataclass
import numpy as np
import pandas as pd
import gurobipy as gp
from gurobipy import GRB, Model, quicksum

In [26]:
# Cell 2: Helper Functions

def haversine_miles(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """Great-circle distance in miles."""
    r = 3958.8
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp = math.radians(lat2 - lat1)
    dl = math.radians(lon2 - lon1)
    a = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return r * (2 * math.atan2(math.sqrt(a), math.sqrt(1 - a)))


def _normalize_areas_columns(areas: pd.DataFrame) -> pd.DataFrame:
    areas = areas.copy()
    if "-5" in areas.columns and "0-5" not in areas.columns:
        areas = areas.rename(columns={"-5": "0-5"})
    return areas


def _normalize_employment_rate(s: pd.Series) -> pd.Series:
    """Accept rate in [0,1] or percent in [0,100] — converts to [0,1]."""
    s = pd.to_numeric(s, errors="coerce")
    if s.dropna().empty:
        return s.fillna(0.0)
    if s.dropna().quantile(0.95) > 1.5:
        return (s / 100.0).fillna(0.0)
    return s.fillna(0.0)

In [27]:
# Cell 3: Result Dataclass

@dataclass
class Model2Result:
    status: int
    objective: float | None
    expansions: pd.DataFrame
    new_facilities: pd.DataFrame
    model: Model

In [28]:
# Cell 4: solve_model2() — Full Model Definition

def solve_model2(
    areas: pd.DataFrame,
    existing_facilities: pd.DataFrame,
    potential_locations: pd.DataFrame,
    facilities_type: pd.DataFrame,
    min_distance_miles: float = 0.06,
    time_limit_sec: int = 600,
    mip_gap: float = 0.01,
    verbose: bool = True,
) -> Model2Result:

    # --- Data Prep ---
    areas = _normalize_areas_columns(areas.copy())
    existing = existing_facilities.copy()
    locations = potential_locations.copy()
    ftypes = facilities_type.copy()

    for df in (areas, existing, locations):
        df["zipcode"] = df["zipcode"].astype(str)

    # --- Column Validation ---
    req_areas = {"zipcode", "average income", "employment rate", "0-5", "6-12", "total_capacity", "0-5 years capacity"}
    req_existing = {"zipcode", "total_capacity", "0-5 years capacity", "latitude", "longitude"}
    req_locations = {"zipcode", "latitude", "longitude"}
    req_ftypes = {"facility_type", "slots total", "slots 0-5 years", "cost"}

    missing = {
        "areas": req_areas - set(areas.columns),
        "existing_facilities": req_existing - set(existing.columns),
        "potential_locations": req_locations - set(locations.columns),
        "facilities_type": req_ftypes - set(ftypes.columns),
    }
    bad = {k: v for k, v in missing.items() if v}
    if bad:
        raise ValueError(f"Missing required columns: {bad}")

    # --- Type Coercion ---
    if "location_id" not in locations.columns:
        locations = locations.reset_index().rename(columns={"index": "location_id"})

    for c in ["average income", "0-5", "6-12", "total_capacity", "0-5 years capacity"]:
        areas[c] = pd.to_numeric(areas[c], errors="coerce").fillna(0.0)
    areas["employment rate"] = _normalize_employment_rate(areas["employment rate"])

    for c in ["total_capacity", "0-5 years capacity", "latitude", "longitude"]:
        existing[c] = pd.to_numeric(existing[c], errors="coerce")
    for c in ["latitude", "longitude"]:
        locations[c] = pd.to_numeric(locations[c], errors="coerce")

    existing = existing.dropna(subset=["latitude", "longitude"]).copy()
    locations = locations.dropna(subset=["latitude", "longitude"]).copy()

    if areas["zipcode"].duplicated().any():
        dup = sorted(areas.loc[areas["zipcode"].duplicated(), "zipcode"].astype(str).unique().tolist())
        raise ValueError(f"areas has duplicate zipcode rows. Examples: {dup[:10]}")

    # --- Demand Calculations ---
    areas["children_0_12"] = areas["0-5"] + areas["6-12"]
    areas["high_demand"] = (
        (areas["employment rate"] >= 0.60) | (areas["average income"] <= 60000)
    ).astype(int)
    areas["required_total_slots"] = np.where(
        areas["high_demand"] == 1,
        0.5 * areas["children_0_12"],
        (1.0 / 3.0) * areas["children_0_12"],
    )
    areas["required_under5_slots"] = (2.0 / 3.0) * areas["0-5"]

    # --- Index Setup ---
    F = existing.index.tolist()
    L = locations["location_id"].tolist()
    T = sorted(ftypes["facility_type"].tolist())

    ft = ftypes.set_index("facility_type")
    new_slots_total = ft["slots total"].to_dict()
    new_slots_u5   = ft["slots 0-5 years"].to_dict()
    new_cost       = ft["cost"].to_dict()

    f_zip = existing["zipcode"].to_dict()
    f_n   = existing["total_capacity"].fillna(0.0).to_dict()
    f_lat = existing["latitude"].to_dict()
    f_lon = existing["longitude"].to_dict()

    loc_zip = dict(zip(locations["location_id"], locations["zipcode"]))
    loc_lat = dict(zip(locations["location_id"], locations["latitude"]))
    loc_lon = dict(zip(locations["location_id"], locations["longitude"]))

    zips = sorted(areas["zipcode"].unique().tolist())
    fac_by_zip = {z: [] for z in zips}
    loc_by_zip = {z: [] for z in zips}
    for f in F:
        z = f_zip[f]
        if z in fac_by_zip:
            fac_by_zip[z].append(f)
    for l in L:
        z = loc_zip[l]
        if z in loc_by_zip:
            loc_by_zip[z].append(l)

    # --- Build Gurobi Model ---
    m = Model("IEOR4004_Model2_Realistic")
    m.Params.OutputFlag = 1 if verbose else 0
    m.Params.TimeLimit  = time_limit_sec
    m.Params.MIPGap     = mip_gap

    x    = m.addVars(F, lb=0.0, vtype=GRB.CONTINUOUS, name="expand_slots_total")
    x_u5 = m.addVars(F, lb=0.0, vtype=GRB.CONTINUOUS, name="expand_slots_0_5")
    cexp = m.addVars(F, lb=0.0, vtype=GRB.CONTINUOUS, name="expand_cost")
    b1   = m.addVars(F, vtype=GRB.BINARY, name="seg_0_10")
    b2   = m.addVars(F, vtype=GRB.BINARY, name="seg_10_15")
    b3   = m.addVars(F, vtype=GRB.BINARY, name="seg_15_20")
    y    = m.addVars(L, T, vtype=GRB.BINARY, name="build")

    # --- Piecewise Expansion Costs ---
    eps = 1e-6
    for f in F:
        n = float(f_n[f])
        if n <= 0:
            m.addConstr(x[f] == 0);  m.addConstr(x_u5[f] == 0)
            m.addConstr(b1[f] + b2[f] + b3[f] == 0);  m.addConstr(cexp[f] == 0)
            continue

        m.addConstr(b1[f] + b2[f] + b3[f] <= 1,  name=f"one_segment[{f}]")
        m.addConstr(x_u5[f] <= x[f],              name=f"u5_le_total[{f}]")
        m.addConstr(x[f] <= 0.10*n*b1[f] + 0.15*n*b2[f] + 0.20*n*b3[f], name=f"seg_ub[{f}]")
        m.addConstr(x[f] >= eps*b1[f] + 0.10*n*b2[f] + 0.15*n*b3[f],    name=f"seg_lb[{f}]")

        a1 = (20000.0 + 200.0*n) / n
        a2 = (20000.0 + 400.0*n) / n
        a3 = (20000.0 + 1000.0*n) / n
        m_big = a3 * (0.20*n) + 100.0*(0.20*n) + 1.0

        m.addConstr(cexp[f] >= a1*x[f] + 100.0*x_u5[f] - m_big*(1-b1[f]), name=f"cost1_lb[{f}]")
        m.addConstr(cexp[f] >= a2*x[f] + 100.0*x_u5[f] - m_big*(1-b2[f]), name=f"cost2_lb[{f}]")
        m.addConstr(cexp[f] >= a3*x[f] + 100.0*x_u5[f] - m_big*(1-b3[f]), name=f"cost3_lb[{f}]")
        m.addConstr(cexp[f] <= m_big*(b1[f]+b2[f]+b3[f]),                  name=f"cost_zero_if_noexp[{f}]")

    for l in L:
        m.addConstr(quicksum(y[l, t] for t in T) <= 1, name=f"one_type_per_loc[{l}]")

    # --- Distance Constraints (within zip) ---
    for z in zips:
        locs = loc_by_zip.get(z, [])
        facs = fac_by_zip.get(z, [])
        for i in range(len(locs)):
            for j in range(i+1, len(locs)):
                li, lj = locs[i], locs[j]
                if haversine_miles(loc_lat[li], loc_lon[li], loc_lat[lj], loc_lon[lj]) < min_distance_miles:
                    m.addConstr(quicksum(y[li,t] for t in T) + quicksum(y[lj,t] for t in T) <= 1,
                                name=f"loc_loc_dist[{z}_{li}_{lj}]")
        for l in locs:
            if any(haversine_miles(loc_lat[l], loc_lon[l], f_lat[f], f_lon[f]) < min_distance_miles for f in facs):
                m.addConstr(quicksum(y[l,t] for t in T) == 0, name=f"block_loc_near_existing[{z}_{l}]")

    # --- Coverage Constraints ---
    areas_i = areas.set_index("zipcode")
    for z in zips:
        row = areas_i.loc[z]
        total_expr = (
            float(row["total_capacity"])
            + quicksum(x[f] for f in fac_by_zip.get(z, []))
            + quicksum(new_slots_total[t]*y[l,t] for l in loc_by_zip.get(z,[]) for t in T)
        )
        under5_expr = (
            float(row["0-5 years capacity"])
            + quicksum(x_u5[f] for f in fac_by_zip.get(z, []))
            + quicksum(new_slots_u5[t]*y[l,t] for l in loc_by_zip.get(z,[]) for t in T)
        )
        m.addConstr(total_expr  >= float(row["required_total_slots"]), name=f"desert_elimination[{z}]")
        m.addConstr(under5_expr >= float(row["required_under5_slots"]), name=f"under5_requirement[{z}]")

    # --- Objective ---
    m.setObjective(
        quicksum(cexp[f] for f in F) + quicksum(new_cost[t]*y[l,t] for l in L for t in T),
        GRB.MINIMIZE,
    )

    m.optimize()

    # --- Extract Results ---
    obj = float(m.ObjVal) if m.SolCount > 0 else None

    exp_rows = []
    if m.SolCount > 0:
        for f in F:
            xv = x[f].X
            if xv > 1e-6:
                seg = 1 if b1[f].X > 0.5 else 2 if b2[f].X > 0.5 else 3 if b3[f].X > 0.5 else 0
                exp_rows.append({"facility_index": f, "zipcode": f_zip[f],
                                 "expanded_slots_total": xv, "expanded_slots_0_5": x_u5[f].X,
                                 "expansion_cost": cexp[f].X, "segment": seg})

    new_rows = []
    if m.SolCount > 0:
        for l in L:
            for t in T:
                if y[l,t].X > 0.5:
                    new_rows.append({"location_id": l, "zipcode": loc_zip[l], "facility_type": t,
                                     "slots_total": new_slots_total[t], "slots_0_5": new_slots_u5[t],
                                     "cost": new_cost[t], "latitude": loc_lat[l], "longitude": loc_lon[l]})

    return Model2Result(
        status=int(m.Status), objective=obj,
        expansions=pd.DataFrame(exp_rows),
        new_facilities=pd.DataFrame(new_rows),
        model=m,
    )

In [29]:
# Cell 5: Run the Model
res = solve_model2(
    areas=areas,
    existing_facilities=existing_facilities,
    potential_locations=potential_locations,
    facilities_type=facilities_type,
    min_distance_miles=0.06,
    time_limit_sec=600,
    mip_gap=0.01,
    verbose=True,
)

Set parameter OutputFlag to value 1
Set parameter TimeLimit to value 600
Set parameter MIPGap to value 0.01
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[arm] - Darwin 24.6.0 24G517)

CPU model: Apple M4 Pro
Thread count: 14 physical cores, 14 logical processors, using up to 14 threads

Non-default parameters:
TimeLimit  600
MIPGap  0.01



GurobiError: Model too large for size-limited license; visit https://gurobi.com/unrestricted for more information

In [ ]:
# ============================================================
# Cell 6: Display Results
# ============================================================

print("-" * 50)
print(f"✅ Gurobi Solve Status (2 = OPTIMAL): {res.status}")
if res.objective is not None:
    print(f"💰 Optimal Minimum Funding Required: ${res.objective:,.2f}")
print(f"🏗️  Total Existing Facilities Expanded: {len(res.expansions)}")
print(f"🏢 Total New Facilities Built:          {len(res.new_facilities)}")
print("-" * 50)

print("\n--- Expansion Details (Top 10) ---")
display(res.expansions.head(10))

print("\n--- New Facilities Details (Top 10) ---")
display(res.new_facilities.head(10))

--------------------------------------------------
✅ Gurobi Solve Status (2 = OPTIMAL): 2
💰 Optimal Minimum Funding Required: $212,555,545.90
🏗️  Total Existing Facilities Expanded: 976
🏢 Total New Facilities Built:          1523
--------------------------------------------------

--- Expansion Details (Top 10) ---


,facility_index,zipcode,expanded_slots_total,expanded_slots_0_5,expansion_cost,segment
0,6,10473,24.000000,24.000000,14000.000000,2
1,19,10462,3.950728,3.950728,1507.726924,1
2,30,11211,28.400000,28.400000,10520.000000,1
3,31,10038,10.000000,10.000000,5000.000000,1
4,32,11104,10.000000,10.000000,5000.000000,1
5,33,10019,5.474910,5.474910,3408.573065,1
6,34,11205,15.150000,15.150000,10575.000000,2
7,51,10033,15.200000,15.200000,6560.000000,1
8,55,10473,29.500000,29.500000,10850.000000,1
9,57,10457,4.000000,4.000000,4000.000000,2



--- New Facilities Details (Top 10) ---


,location_id,zipcode,facility_type,slots_total,slots_0_5,cost,latitude,longitude
0,2,10001,3,400,200,135000,40.750545,-73.997147
1,23,10001,3,400,200,135000,40.752099,-73.993565
2,113,10002,3,400,200,135000,40.709508,-73.980533
3,132,10002,3,400,200,135000,40.717204,-73.978436
4,154,10002,3,400,200,135000,40.708707,-73.995382
5,185,10002,3,400,200,135000,40.719896,-73.994170
6,186,10002,3,400,200,135000,40.723983,-73.987574
7,200,10003,3,400,200,135000,40.733158,-73.998999
8,202,10003,3,400,200,135000,40.722180,-73.981610
9,253,10003,3,400,200,135000,40.731680,-73.997879
